# Supplementary Tables: Ensembl × CAT Gene Concordance

Generates two supplementary TSV tables for publication.

| Table | File | Content |
|-------|------|---------|
| S1 | supp_table_s1_gene_concordance_summary.tsv | Per-gene summary across all 462 assemblies |
| S2 | supp_table_s2_per_assembly_gene_pairs.tsv | Per-assembly × gene-pair detail |

**Input:** Pipeline output directory (set OUTPUT_DIR below)

In [1]:
!pip install -U pandas pyarrow

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import time
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path(os.getenv('HPRC_QC_OUTPUT_DIR',
    '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results'))

QC_DIR = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'
SUPP_DIR = OUTPUT_DIR / 'supplementary_tables'
SUPP_DIR.mkdir(parents=True, exist_ok=True)

# Parquet caches — dramatically faster than re-reading 462 TSV files each run.
# Set FORCE_RELOAD=True to regenerate from raw files (e.g. after pipeline re-run).
CACHE_DIR = SUPP_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RELOAD = False

print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"QC_DIR exists: {QC_DIR.exists()}")
print(f"RESULTS_DIR exists: {RESULTS_DIR.exists()}")
print(f"CACHE_DIR: {CACHE_DIR}")
print(f"FORCE_RELOAD: {FORCE_RELOAD}")

OUTPUT_DIR: /hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results
QC_DIR exists: True
RESULTS_DIR exists: True
CACHE_DIR: /hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results/supplementary_tables/cache
FORCE_RELOAD: False


## Load per-assembly data

In [3]:
_cache = CACHE_DIR / 'transcript_concordance.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    transcript_concordance_df = pd.read_parquet(_cache)
    print(f"Loaded transcript concordance from cache: {len(transcript_concordance_df):,} rows ({time.time()-t0:.1f}s)")
else:
    transcript_concordance_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_transcript_concordance.tsv'
        try:
            df = pd.read_csv(fp, sep='\t')
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            transcript_concordance_frames.append(df)
        except FileNotFoundError:
            print(f"WARNING: missing file {fp}")
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  transcript concordance: {i+1} assemblies loaded...")

    transcript_concordance_df = pd.concat(transcript_concordance_frames, ignore_index=True)
    transcript_concordance_df.to_parquet(_cache, index=False)
    print(f"Loaded transcript concordance: {len(transcript_concordance_df):,} rows from {len(transcript_concordance_frames)} assemblies → cached")

Loaded transcript concordance from cache: 34,967,917 rows (10.1s)


In [4]:
_cache = CACHE_DIR / 'coding_integrity.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    coding_integrity_df = pd.read_parquet(_cache)
    print(f"Loaded coding integrity from cache: {len(coding_integrity_df):,} rows ({time.time()-t0:.1f}s)")
else:
    coding_integrity_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_coding_integrity.tsv'
        try:
            df = pd.read_csv(fp, sep='\t')
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            coding_integrity_frames.append(df)
        except FileNotFoundError:
            print(f"WARNING: missing file {fp}")
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  coding integrity: {i+1} assemblies loaded...")

    coding_integrity_df = pd.concat(coding_integrity_frames, ignore_index=True)
    coding_integrity_df.to_parquet(_cache, index=False)
    print(f"Loaded coding integrity: {len(coding_integrity_df):,} rows from {len(coding_integrity_frames)} assemblies → cached")

Loaded coding integrity from cache: 8,649,784 rows (1.8s)


In [5]:
import gc

def safe_mode(series):
    m = series.dropna().mode()
    return m.iloc[0] if len(m) > 0 else np.nan

# Gene presence files are skipped entirely — all IDs in the pipeline output are
# assembly-specific, so the only canonical gene key is ensembl_name (HGNC symbol)
# from the rbh files. gp_agg, tc_medians, and rbh_name_lookup are all built in
# cell-8 from rbh_df and merged, while those DataFrames are already in memory.
print("Helpers defined.")

Helpers defined.


In [6]:
_cache = CACHE_DIR / 'divergence.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    divergence_df = pd.read_parquet(_cache)
    print(f"Loaded grch38 divergence from cache: {len(divergence_df):,} rows ({time.time()-t0:.1f}s)")
else:
    divergence_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_grch38_divergence.tsv'
        if not fp.exists():
            continue
        try:
            df = pd.read_csv(fp, sep='\t')
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            divergence_frames.append(df)
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  divergence: {len(divergence_frames)} assemblies loaded...")

    if divergence_frames:
        divergence_df = pd.concat(divergence_frames, ignore_index=True)
        divergence_df.to_parquet(_cache, index=False)
        print(f"Loaded grch38 divergence: {len(divergence_df):,} rows from {len(divergence_frames)} assemblies → cached")
    else:
        divergence_df = pd.DataFrame(columns=['assembly_accession', 'sample_name', 'ensembl_gene_id',
                                               'cat_gene_id', 'gene_name', 'ensembl_biotype',
                                               'ref_biotype', 'divergence_category'])
        print("WARNING: no grch38 divergence files found; divergence_df is empty")

Loaded grch38 divergence from cache: 18,562,662 rows (16.6s)


## Build Supplementary Table S2: Per-assembly gene-pair detail

In [7]:
t0 = time.time()

# Load all RBH gene pair files.
# rbh files use ensembl_id/cat_id and have no assembly_accession —
# inject it from the directory name and rename to pipeline-wide conventions.
rbh_frames = []
for i, accession_dir in enumerate(sorted(RESULTS_DIR.iterdir())):
    if not accession_dir.is_dir():
        continue
    accession = accession_dir.name
    fp = accession_dir / f'{accession}.gene_pairs_rbh.tsv'
    try:
        df = pd.read_csv(fp, sep='\t')
        if df.empty:
            print(f"WARNING: empty file {fp}")
            continue
        df['assembly_accession'] = accession
        df = df.rename(columns={'ensembl_id': 'ensembl_gene_id', 'cat_id': 'cat_gene_id'})
        rbh_frames.append(df)
    except FileNotFoundError:
        print(f"WARNING: missing file {fp}")
    except Exception as e:
        print(f"WARNING: could not read {fp}: {e}")
    if (i + 1) % 50 == 0:
        print(f"  RBH pairs: {len(rbh_frames)} assemblies loaded...")

rbh_df = pd.concat(rbh_frames, ignore_index=True)
del rbh_frames
gc.collect()
print(f"Loaded RBH gene pairs: {len(rbh_df):,} rows from {rbh_df['assembly_accession'].nunique()} assemblies ({time.time()-t0:.1f}s)")

# Keep only RBH pairs (all rows should already have is_rbh=True, but filter to be safe)
rbh_df = rbh_df[rbh_df['is_rbh'] == True].copy()
print(f"After filtering is_rbh=True: {len(rbh_df):,} rows")

# --- Build S2: per-assembly gene-pair detail ---

join_cols = ['assembly_accession', 'ensembl_gene_id', 'cat_gene_id']

# Join transcript concordance
tc_cols = join_cols + [
    'n_ensembl_transcripts', 'n_cat_transcripts',
    'n_ens_exact', 'n_cat_exact',
    'ens_to_cat_concordance_rate', 'cat_to_ens_concordance_rate',
    'avg_jaccard_index',
]
tc_cols = [c for c in tc_cols if c in transcript_concordance_df.columns]
tc_subset = transcript_concordance_df[tc_cols].drop_duplicates(subset=join_cols)
merged = rbh_df.merge(tc_subset, on=join_cols, how='left')
print(f"After joining transcript concordance: {len(merged):,} rows")

# Join coding integrity
ci_cols = join_cols + ['classification', 'start_codon_match', 'stop_codon_match', 'frameshift_detected']
ci_cols = [c for c in ci_cols if c in coding_integrity_df.columns]
ci_subset = coding_integrity_df[ci_cols].drop_duplicates(subset=join_cols)
ci_subset = ci_subset.rename(columns={'classification': 'cds_classification'})
merged = merged.merge(ci_subset, on=join_cols, how='left')
print(f"After joining coding integrity: {len(merged):,} rows")

# Join divergence
if not divergence_df.empty:
    div_cols = [c for c in join_cols + ['divergence_category'] if c in divergence_df.columns]
    div_subset = divergence_df[div_cols].drop_duplicates(subset=join_cols)
    merged = merged.merge(div_subset, on=join_cols, how='left')
else:
    merged['divergence_category'] = None
print(f"After joining divergence: {len(merged):,} rows")

# Compute exact-match pct columns
merged['ens_to_cat_exact_pct'] = np.where(
    merged['n_ensembl_transcripts'] > 0,
    (merged['n_ens_exact'] / merged['n_ensembl_transcripts'] * 100).round(1),
    np.nan
)
merged['cat_to_ens_exact_pct'] = np.where(
    merged['n_cat_transcripts'] > 0,
    (merged['n_cat_exact'] / merged['n_cat_transcripts'] * 100).round(1),
    np.nan
)

s2_cols = [
    'assembly_accession',
    'ensembl_gene_id',
    'cat_gene_id',
    'ensembl_name',   # HGNC symbol — canonical gene identifier
    'cat_name',
    'ensembl_biotype',
    'cat_biotype',
    'frac_ensembl_covered',
    'frac_cat_covered',
    'n_ensembl_transcripts',
    'n_cat_transcripts',
    'n_ens_exact',
    'n_cat_exact',
    'ens_to_cat_exact_pct',
    'cat_to_ens_exact_pct',
    'ens_to_cat_concordance_rate',
    'cat_to_ens_concordance_rate',
    'avg_jaccard_index',
    'cds_classification',
    'start_codon_match',
    'stop_codon_match',
    'frameshift_detected',
    'divergence_category',
]
s2_cols = [c for c in s2_cols if c in merged.columns]
s2_df = merged[s2_cols].copy()

out_s2 = SUPP_DIR / 'supp_table_s2_per_assembly_gene_pairs.tsv'
s2_df.to_csv(out_s2, sep='\t', index=False)
print(f"\nSaved S2: {out_s2}")
print(f"S2 shape: {s2_df.shape} ({time.time()-t0:.1f}s)")
del s2_df

# --- Pre-compute S1 aggregates while rbh_df and merged are in memory ---
# ensembl_name is the HGNC symbol and the only canonical cross-assembly gene key.

n_assemblies_total = rbh_df['assembly_accession'].nunique()

# gp_agg: one row per canonical gene — how many assemblies have it as an RBH pair
gp_agg = (
    rbh_df.groupby('ensembl_name', sort=False)
    .agg(
        n_assemblies_both=('assembly_accession', 'nunique'),
        ensembl_biotype=('ensembl_biotype', safe_mode),
    )
    .reset_index()
    .rename(columns={'ensembl_name': 'gene_name'})
)
gp_agg['n_assemblies_assessed'] = gp_agg['n_assemblies_both']
gp_agg['pct_assemblies_both'] = (gp_agg['n_assemblies_both'] / n_assemblies_total * 100).round(1)
print(f"gp_agg: {len(gp_agg):,} canonical genes ({time.time()-t0:.1f}s)")

# tc_medians: median transcript concordance per canonical gene
# merged already has TC columns joined; ensembl_name is the HGNC symbol
tc_medians = (
    merged.dropna(subset=['ensembl_name'])
    .groupby('ensembl_name')
    .agg(
        median_ens_to_cat_exact_pct=('ens_to_cat_exact_pct', 'median'),
        median_cat_to_ens_exact_pct=('cat_to_ens_exact_pct', 'median'),
        median_ens_to_cat_concordance_rate=('ens_to_cat_concordance_rate', 'median'),
        median_cat_to_ens_concordance_rate=('cat_to_ens_concordance_rate', 'median'),
    )
    .reset_index()
    .round(1)
    .rename(columns={'ensembl_name': 'gene_name'})
)
print(f"tc_medians: {len(tc_medians):,} genes ({time.time()-t0:.1f}s)")

# rbh_name_lookup: (assembly_accession, ensembl_gene_id) → gene_name
# Used in cell-10 to add canonical gene_name to coding_integrity rows.
rbh_name_lookup = (
    rbh_df[['assembly_accession', 'ensembl_gene_id', 'ensembl_name']]
    .drop_duplicates(['assembly_accession', 'ensembl_gene_id'])
    .rename(columns={'ensembl_name': 'gene_name'})
    .reset_index(drop=True)
)
print(f"rbh_name_lookup: {len(rbh_name_lookup):,} entries ({time.time()-t0:.1f}s)")

del merged, rbh_df, transcript_concordance_df
gc.collect()
print("Large DataFrames freed.")

  RBH pairs: 50 assemblies loaded...
  RBH pairs: 100 assemblies loaded...
  RBH pairs: 150 assemblies loaded...
  RBH pairs: 200 assemblies loaded...
  RBH pairs: 250 assemblies loaded...
  RBH pairs: 300 assemblies loaded...
  RBH pairs: 350 assemblies loaded...
  RBH pairs: 400 assemblies loaded...
  RBH pairs: 450 assemblies loaded...
Loaded RBH gene pairs: 34,967,917 rows from 462 assemblies (198.2s)
After filtering is_rbh=True: 34,967,917 rows
After joining transcript concordance: 34,967,917 rows
After joining coding integrity: 34,967,917 rows
After joining divergence: 34,967,917 rows

Saved S2: /hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results/supplementary_tables/supp_table_s2_per_assembly_gene_pairs.tsv
S2 shape: (34967917, 23) (766.9s)
gp_agg: 40,824 canonical genes (791.9s)
tc_medians: 40,824 genes (825.1s)
rbh_name_lookup: 34,967,917 entries (838.5s)
Large DataFrames freed.


## Build Supplementary Table S1: Gene-level concordance summary

In [10]:
t0 = time.time()

# --- CDS integrity per canonical gene ---
# coding_integrity_df has (assembly_accession, ensembl_gene_id) but no gene_name.
# Use rbh_name_lookup to add canonical gene_name (HGNC symbol); inner join so we
# only keep genes that appear as RBH pairs.
ci_with_gene = coding_integrity_df.merge(
    rbh_name_lookup, on=['assembly_accession', 'ensembl_gene_id'], how='inner'
)
print(f"ci_with_gene: {len(ci_with_gene):,} rows ({time.time()-t0:.1f}s)")

# Normalise boolean columns
for col in ['start_codon_match', 'stop_codon_match', 'frameshift_detected']:
    if col in ci_with_gene.columns:
        ci_with_gene[col] = ci_with_gene[col].map(
            {True: True, False: False, 'True': True, 'False': False, 1: True, 0: False}
        )

ci_agg = ci_with_gene.groupby('gene_name').agg(
    n_assemblies_cds_assessed=('assembly_accession', 'nunique'),
    _n_start=('start_codon_match', 'count'),
    _sum_start=('start_codon_match', 'sum'),
    _n_stop=('stop_codon_match', 'count'),
    _sum_stop=('stop_codon_match', 'sum'),
    _n_fs=('frameshift_detected', 'count'),
    _sum_fs=('frameshift_detected', 'sum'),
).reset_index()
del ci_with_gene
gc.collect()

ci_agg['pct_start_codon_match'] = np.where(
    ci_agg['_n_start'] > 0, (ci_agg['_sum_start'] / ci_agg['_n_start'] * 100).round(1), np.nan)
ci_agg['pct_stop_codon_match'] = np.where(
    ci_agg['_n_stop'] > 0, (ci_agg['_sum_stop'] / ci_agg['_n_stop'] * 100).round(1), np.nan)
ci_agg['pct_frameshift_detected'] = np.where(
    ci_agg['_n_fs'] > 0, (ci_agg['_sum_fs'] / ci_agg['_n_fs'] * 100).round(1), np.nan)
ci_agg = ci_agg[['gene_name', 'n_assemblies_cds_assessed',
                   'pct_start_codon_match', 'pct_stop_codon_match', 'pct_frameshift_detected']]
print(f"ci_agg: {len(ci_agg):,} genes ({time.time()-t0:.1f}s)")

# --- Divergence per canonical gene ---
# divergence_df has a gene_name column directly (HGNC symbol from the grch38 divergence pipeline).
if not divergence_df.empty and 'gene_name' in divergence_df.columns:
    div_base = divergence_df.dropna(subset=['gene_name', 'divergence_category'])
    div_counts = (
        div_base.groupby(['gene_name', 'divergence_category'])
        .size().unstack(fill_value=0).reset_index()
    )
    for cat in ['both_agree_reference', 'both_agree_diverged', 'ensembl_specific', 'cat_specific']:
        if cat not in div_counts.columns:
            div_counts[cat] = 0
    div_counts['_n_div_total'] = div_counts[
        ['both_agree_reference', 'both_agree_diverged', 'ensembl_specific', 'cat_specific']
    ].sum(axis=1)
    for cat in ['both_agree_reference', 'both_agree_diverged', 'ensembl_specific', 'cat_specific']:
        div_counts[f'pct_{cat}'] = np.where(
            div_counts['_n_div_total'] > 0,
            (div_counts[cat] / div_counts['_n_div_total'] * 100).round(1), np.nan)
    div_mode = (
        div_base.groupby('gene_name')['divergence_category']
        .agg(safe_mode).reset_index()
    )
    div_mode.columns = ['gene_name', 'predominant_divergence_category']
    div_final = div_counts[['gene_name', 'pct_both_agree_reference', 'pct_both_agree_diverged',
                              'pct_ensembl_specific', 'pct_cat_specific']]
    div_final = div_final.merge(div_mode, on='gene_name', how='left')
else:
    div_final = pd.DataFrame(columns=['gene_name', 'predominant_divergence_category',
                                       'pct_both_agree_reference', 'pct_both_agree_diverged',
                                       'pct_ensembl_specific', 'pct_cat_specific'])
print(f"div_final: {len(div_final):,} genes ({time.time()-t0:.1f}s)")

# --- Assemble S1 ---
# All aggregates are keyed by gene_name (HGNC symbol from ensembl_name in rbh files).
s1_df = gp_agg.copy()
s1_df = s1_df.merge(tc_medians, on='gene_name', how='left')
s1_df = s1_df.merge(ci_agg, on='gene_name', how='left')
s1_df = s1_df.merge(div_final, on='gene_name', how='left')
s1_df['predominant_divergence_category'] = s1_df['predominant_divergence_category'].fillna('N/A')

s1_col_order = [
    'gene_name',
    'ensembl_biotype',
    'n_assemblies_assessed',
    'n_assemblies_both',
    'pct_assemblies_both',
    'median_ens_to_cat_exact_pct',
    'median_cat_to_ens_exact_pct',
    'median_ens_to_cat_concordance_rate',
    'median_cat_to_ens_concordance_rate',
    'n_assemblies_cds_assessed',
    'pct_start_codon_match',
    'pct_stop_codon_match',
    'pct_frameshift_detected',
    'predominant_divergence_category',
    'pct_both_agree_reference',
    'pct_both_agree_diverged',
    'pct_ensembl_specific',
    'pct_cat_specific',
]
s1_col_order = [c for c in s1_col_order if c in s1_df.columns]
s1_df = s1_df[s1_col_order].sort_values('gene_name').reset_index(drop=True)

out_s1 = SUPP_DIR / 'supp_table_s1_gene_concordance_summary.tsv'
s1_df.to_csv(out_s1, sep='\t', index=False)
print(f"\nSaved S1: {out_s1}")
print(f"S1 shape: {s1_df.shape} ({time.time()-t0:.1f}s total)")
s1_df.head(30)

ci_with_gene: 8,649,784 rows (18.4s)
ci_agg: 19,515 genes (21.6s)
div_final: 40,821 genes (40.7s)

Saved S1: /hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results/supplementary_tables/supp_table_s1_gene_concordance_summary.tsv
S1 shape: (40824, 18) (41.2s total)


,gene_name,ensembl_biotype,n_assemblies_assessed,n_assemblies_both,pct_assemblies_both,median_ens_to_cat_exact_pct,median_cat_to_ens_exact_pct,median_ens_to_cat_concordance_rate,median_cat_to_ens_concordance_rate,n_assemblies_cds_assessed,pct_start_codon_match,pct_stop_codon_match,pct_frameshift_detected,predominant_divergence_category,pct_both_agree_reference,pct_both_agree_diverged,pct_ensembl_specific,pct_cat_specific
0,5S_rRNA,rRNA,344,344,74.5,100.0,100.0,1.0,1.0,NaN,NaN,NaN,NaN,both_agree_diverged,28.7,71.3,0.0,0.0
1,5_8S_rRNA,rRNA,344,344,74.5,100.0,100.0,1.0,1.0,NaN,NaN,NaN,NaN,both_agree_diverged,3.7,96.3,0.0,0.0
2,7SK,misc_RNA,344,344,74.5,100.0,100.0,1.0,1.0,NaN,NaN,NaN,NaN,both_agree_diverged,14.9,85.1,0.0,0.0
3,A1BG,protein_coding,441,441,95.5,100.0,55.6,1.0,0.6,441.0,99.3,99.3,0.0,cat_specific_divergence,0.0,100.0,0.0,0.0
4,A1BG-AS1,lncRNA,218,218,47.2,100.0,69.2,1.0,0.7,NaN,NaN,NaN,NaN,cat_specific_divergence,96.6,3.4,0.0,0.0
5,A1CF,protein_coding,458,458,99.1,100.0,100.0,1.0,1.0,458.0,100.0,100.0,0.0,both_agree_reference,97.8,2.2,0.0,0.0
6,A2M,protein_coding,458,458,99.1,100.0,100.0,1.0,1.0,458.0,100.0,100.0,0.0,both_agree_reference,100.0,0.0,0.0,0.0
7,A2M-AS1,lncRNA,439,439,95.0,100.0,100.0,1.0,1.0,NaN,NaN,NaN,NaN,both_agree_reference,100.0,0.0,0.0,0.0
8,A2ML1,protein_coding,458,458,99.1,100.0,100.0,1.0,1.0,458.0,100.0,100.0,5.0,both_agree_diverged,28.6,71.4,0.0,0.0
9,A2ML1-AS1,lncRNA,458,458,99.1,100.0,100.0,1.0,1.0,NaN,NaN,NaN,NaN,both_agree_reference,78.7,21.3,0.0,0.0


## Summary

In [9]:
print(f"Table S1: {len(s1_df):,} genes \u00d7 {len(s1_df.columns)} columns")
print(f"Table S2: {len(s2_df):,} gene-pair records \u00d7 {len(s2_df.columns)} columns")
print(f"\nFiles written to: {SUPP_DIR}")
for f in sorted(SUPP_DIR.glob('*.tsv')):
    size = f.stat().st_size / 1024**2
    print(f"  {f.name}: {size:.1f} MB")

Table S1: 40,824 genes × 18 columns


NameError: name 's2_df' is not defined